# Kaggle ComfyUI + LTX 2.3 GGUF
Model: Unsloth Q3_K_M | Workflows: T2V, I2V, ICLoRA, Lipdub, Motion Track
---

In [ ]:
MODEL_VARIANT="Q3_K_M";DOWNLOAD_MODELS=True;DOWNLOAD_WORKFLOWS=True;INCLUDE_ICLORA=True;LOW_VRAM_MODE=True
print(f"Config: {MODEL_VARIANT}")

In [ ]:
import os,sys,subprocess,shutil,time,threading
from pathlib import Path
WORKING=Path("/kaggle/working");COMFY=WORKING/"ComfyUI";VENV=WORKING/"venv"
start=time.time();print("Installing...")
if not VENV.exists():
    subprocess.run([sys.executable,"-m","pip","install","-q","virtualenv"],check=True)
    subprocess.run(["virtualenv",str(VENV),"-p","/usr/bin/python3.10"],check=True)
PY=str(VENV/"bin"/"python3.10");PIP=[PY,"-m","pip","install","-q"]
subprocess.run([*PIP,"torch","torchvision","torchaudio","--index-url","https://download.pytorch.org/whl/cu124"],check=True)
subprocess.run([*PIP,"requests","einops","opencv-python"],check=True)
if not COMFY.exists():
    subprocess.run(["git","clone","--branch","ComfyUI_ltx_2_3_compliant_19_03_2026","https://github.com/Isi-dev/ComfyUI",str(COMFY)],check=True)
    subprocess.run([*PIP,"-r",str(COMFY/"requirements.txt")],check=True)
print("ComfyUI OK")
for n,u in {"ComfyUI-Manager":"https://github.com/Comfy-Org/ComfyUI-Manager","ComfyUI-GGUF":"https://github.com/city96/ComfyUI-GGUF","ComfyUI-LTXVideo":"https://github.com/Lightricks/ComfyUI-LTXVideo","rgthree-comfy":"https://github.com/rgthree/rgthree-comfy.git"}.items():
    p=COMFY/"custom_nodes"/n
    if not p.exists():
        subprocess.run(["git","clone",u,str(p)],capture_output=True,check=False)
        req=p/"requirements.txt"
        if req.exists():subprocess.run([*PIP,"-r",str(req)],capture_output=True,check=False)
for d in ["unet","vae","text_encoders","loras","latent_upscale_models"]:
    dst=Path(f"/tmp/models/{d}");dst.mkdir(parents=True,exist_ok=True)
    src=COMFY/"models"/d
    if not src.exists():src.mkdir(parents=True,exist_ok=True)
    for f in list(src.iterdir()):
        if f.is_file():f.unlink()
        elif f.is_symlink():f.unlink()
        else:shutil.rmtree(str(f))
    if not src.is_symlink():
        shutil.rmtree(str(src))
        src.symlink_to(dst)
print(f"Env done ({time.time()-start:.0f}s)")

In [ ]:
import urllib.request,time
from pathlib import Path
MDIR=Path("/tmp/models")
URLS={"Model":f"https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-{MODEL_VARIANT}.gguf"}
DEST={"Model":MDIR/"unet"/f"ltx-2.3-22b-dev-{MODEL_VARIANT}.gguf"}
subprocess.run(["df","-h","/tmp","/kaggle/working"])
if DOWNLOAD_MODELS:
    t0=time.time()
    for name,url in URLS.items():
        p=DEST[name]
        if p.exists() and p.stat().st_size>1e6:print(f"  OK {name} ({p.stat().st_size/1e9:.1f} GB)");continue
        print(f"  DL {name}...",end=" ",flush=True)
        urllib.request.urlretrieve(url,p)
        print(f"{p.stat().st_size/1e9:.1f} GB")
    t=sum(f.stat().st_size for f in MDIR.rglob("*")if f.is_file())/1e9
    print(f"Total: {t:.1f} GB ({time.time()-t0:.0f}s)")

In [ ]:
from pathlib import Path
wd=Path("/kaggle/working/workflows");wd.mkdir(exist_ok=True)
WFS={"T2V+I2V":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_T2V_I2V_Single_Stage_Distilled_Full.json","ICLoRA_Union":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Union_Control_Distilled.json","Lipdub":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Lipdub_Two_Stage_Distilled.json","Motion_Track":"https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/example_workflows/2.3/LTX-2.3_ICLoRA_Motion_Track_Distilled.json"}
if DOWNLOAD_WORKFLOWS:
 for n,u in WFS.items():
  d=wd/f"{n}.json"
  if not d.exists():
   try:urllib.request.urlretrieve(u,d);print(f"  {n}")
   except:print(f"  FAIL {n}")
 print("Done")

In [ ]:
import urllib.request,subprocess,os,time,threading
from pathlib import Path
COMFY=Path("/kaggle/working/ComfyUI")
PY=str(Path("/kaggle/working/venv/bin/python3.10"))
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,index","--format=csv,noheader"])

# Start ComfyUI
subprocess.Popen([PY,str(COMFY/"main.py"),"--listen","127.0.0.1","--port","8188","--highvram"],
    stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
print("ComfyUI started")

# Wait for API
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system/stats",timeout=3)
        print(f"API ready! ({i*2}s)");break
    except:pass

# SSH tunnel (khong dung pinggy.py - no restart ComfyUI)
print("\nCreating tunnel...")
URL_FILE="/kaggle/working/url.txt"

def tunnel():
    p=subprocess.Popen(["ssh","-p","443","-o","StrictHostKeyChecking=no","-R0:localhost:8188","a.pinggy.io"],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,universal_newlines=True)
    for line in p.stdout:
        if "https://" in line:
            i=line.find("https://")
            url=line[i:].strip().split()[0]
            open(URL_FILE,"w").write(url)
            print(f"\nTUNNEL URL: {url}")
            break

threading.Thread(target=tunnel,daemon=True).start()

# Wait for URL
TUNNEL_URL=None
for i in range(60):
    time.sleep(3)
    if TUNNEL_URL:break
    try:
        TUNNEL_URL=open(URL_FILE).read().strip()
    except:pass

if TUNNEL_URL:
    print(f"URL: {TUNNEL_URL}")
    print(f"POST {TUNNEL_URL}/prompt")
else:
    print("No URL yet")